# Laborator 5 · Analiza și explorarea tiparelor (Pattern Mining)
**Dataset:** Online Retail II (UCI) · **Metodologie:** CRISP-DM

| Fază CRISP-DM | Secțiune |
|---|---|
| 1. Înțelegerea sarcinii | 0 |
| 2. Înțelegerea datelor | 2 |
| 3. Pregătirea datelor | 3 |
| 4. Modelare | 4 (Apriori, FP-Growth, ECLAT, reguli, reguli cantitative, tipare secvențiale) |
| 5. Evaluare | 5 |
| 6. Implementare | 6 (+ pagina Streamlit cu recomandări) |

## 0. Înțelegerea sarcinii (Business understanding)
**Întrebare:** ce produse se cumpără împreună și ce reguli de tip „dacă A, atunci B” putem folosi pentru recomandări, plasarea produselor și pachete promoționale? (aplicații tipice: analiza coșului de cumpărături, sisteme de recomandare)

**Concepte:** seturi de itemi frecvenți, reguli de asociere `A ⇒ B`, tipare secvențiale, reguli cantitative. Metrici: **support** (cât de des apare), **confidence** (`P(B | A)`), **lift** (`confidence / support(B)`; 1 = independență, > 1 = asociere pozitivă).
**Provocări** (le tratăm explicit): numărul mare de tipare posibile, setarea pragurilor, utilitatea practică a regulilor, costul computațional.

**Criterii de succes:**
1. Cei trei algoritmi (Apriori, FP-Growth, ECLAT) dau **aceleași** seturi frecvente (corectitudine), la costuri de timp diferite.
2. Regulile **rezistă în timp**: mining pe istoric, validare pe ultimele 90 de zile (nevăzute).
3. Regulile sunt **utile**: ca recomandări, bat o bază simplă (produsele populare) pe un test „ascunde un produs din coș și ghicește-l”.

## 1. Setup

In [ ]:
import json
import math
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth

from baza_date import (ARTEFACTE, SEED, construieste_clienti, curata, descrieri_produse, incarca_brut, salveaza_fig)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["PYTHONWARNINGS"] = "ignore"
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.3f}".format)

try:                                   # `display` există în Jupyter/VS Code; în terminal îl înlocuim cu print
    display
except NameError:
    def display(obj):
        print(obj)

MAX_LEN = 3                            # lungimea maximă a unui set de itemi (2 = perechi, 3 = triplete)
MIN_CONF = 0.5                         # „dacă A, atunci B” cel puțin jumătate din dăți
MAX_REGULI = 1000                      # cât poate verifica un analist (criteriu pentru alegerea suportului)
GRILA_SUPORT = [0.005, 0.01, 0.02, 0.03, 0.05]

## 2. Înțelegerea datelor (Data understanding)
### 2.1 Încărcare, curățare, coșuri
Aceeași curățare ca în Laboratoarele 1–2 (`baza_date.py`). O **tranzacție** (coș) = o factură; conținutul ei = mulțimea codurilor de produs (fiecare produs o singură dată pe factură). Folosim și facturile fără `Customer ID`: pentru coșuri nu avem nevoie de client.

In [ ]:
df = incarca_brut()
tranzactii, retururi, administrative, ajustari_stoc, contabilitate = curata(df)
contabilitate.to_csv(ARTEFACTE / "curatare.csv", index=False)
clienti, CUTOFF, istoric_v, viitor_v = construieste_clienti(tranzactii, retururi)
descriere = descrieri_produse(tranzactii)
display(contabilitate)

cosuri = tranzactii[["Invoice", "StockCode", "InvoiceDate"]].drop_duplicates(["Invoice", "StockCode"])
print(f"Facturi (coșuri): {cosuri['Invoice'].nunique():,} | produse distincte: {cosuri['StockCode'].nunique():,} | rânduri factură–produs: {len(cosuri):,}")

### 2.2 Profilul coșurilor
Ce vrem să aflăm: cât de mari sunt coșurile (coșurile cu un singur produs nu pot forma perechi) și cât de **rară** e informația (câte produse sunt cumpărate des). Asta decide ce praguri de suport au sens.

In [ ]:
dim_cos = cosuri.groupby("Invoice").size()
frecv_item = cosuri.groupby("StockCode").size().sort_values(ascending=False)
profil = pd.DataFrame({"valoare": [
    f"{len(dim_cos):,}", f"{len(frecv_item):,}", f"{dim_cos.median():.0f}", f"{dim_cos.mean():.1f}", f"{dim_cos.max():,}", f"{(dim_cos == 1).mean():.1%}",
    f"{(frecv_item / len(dim_cos) >= 0.01).sum()}", f"{(frecv_item / len(dim_cos) < 0.001).mean():.1%}",
    f"{frecv_item.iloc[:10].sum() / len(cosuri):.1%}"]},
    index=["facturi", "produse distincte", "produse pe factură (mediană)", "produse pe factură (medie)", "produse pe factură (max)",
           "facturi cu un singur produs", "produse cu suport ≥ 1%", "produse cu suport < 0,1% (coada lungă)", "ponderea celor mai frecvente 10 produse"])
profil.to_csv(ARTEFACTE / "lab5_profil.csv")
display(profil)

top_items = frecv_item.head(10).to_frame("facturi")
top_items["suport"] = top_items["facturi"] / len(dim_cos)
top_items.insert(0, "produs", descriere.reindex(top_items.index).values)
top_items.to_csv(ARTEFACTE / "lab5_top_produse.csv")
display(top_items)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
p99 = dim_cos.quantile(0.99)
dim_cos[dim_cos <= p99].hist(bins=40, ax=ax[0]); ax[0].set_title(f"Produse pe factură (facturile ≤ percentila 99 = {p99:.0f})")
ax[1].loglog(range(1, len(frecv_item) + 1), frecv_item.values); ax[1].set_title("Frecvența produselor (scară log–log)")
ax[1].set_xlabel("rang"); ax[1].set_ylabel("nr. facturi")
plt.tight_layout()
salveaza_fig("lab5_profil")

## 3. Pregătirea datelor (Data preparation)
- **Împărțire în timp:** mining pe facturile **dinainte** de data de tăiere (istoric), validare pe cele **de după** (ultimele 90 de zile). Un tipar real trebuie să se mențină și pe date noi.
- **Matrice booleană** coș × produs. Păstrăm doar produsele cu suport ≥ cel mai mic prag din grilă (0,5%): un produs mai rar nu poate apărea în niciun set frecvent la pragurile testate.
- **Alegerea suportului minim** (setarea pragurilor e punctul sensibil al metodei): rulăm FP-Growth pentru fiecare prag din grilă și numărăm seturile și regulile. **Regulă:** cel mai mic suport pentru care rămân cel mult 1.000 de reguli (cu confidence ≥ 0,5 și lift > 1), număr pe care un analist îl poate verifica.

In [ ]:
produse_pastrate = frecv_item[frecv_item / len(dim_cos) >= min(GRILA_SUPORT)].index
sel = cosuri[cosuri["StockCode"].isin(produse_pastrate)]
cod_f, inv_idx = pd.factorize(sel["Invoice"])
cod_p, prod_idx = pd.factorize(sel["StockCode"])
mat = np.zeros((len(inv_idx), len(prod_idx)), dtype=bool)
mat[cod_f, cod_p] = True
onehot = pd.DataFrame(mat, columns=prod_idx, index=inv_idx)
data_cos = sel.groupby("Invoice")["InvoiceDate"].first().reindex(inv_idx)
tren = (data_cos < CUTOFF).to_numpy()
X_tr, X_te = onehot[tren], onehot[~tren]
print(f"Coșuri train (istoric): {len(X_tr):,} | test (după {CUTOFF.date()}): {len(X_te):,} | produse (coloane): {onehot.shape[1]}")


def genereaza_reguli(sup, min_conf, consecvente=None, min_lift=1.0):
    """Reguli A ⇒ c (consecvent cu un singur item) din dicționarul de suporturi {frozenset: suport}.
    support = sup(A ∪ c); confidence = sup(A ∪ c) / sup(A); lift = confidence / sup(c). Toate subseturile unui set frecvent sunt frecvente."""
    rez = []
    for S, s in sup.items():
        if len(S) < 2:
            continue
        for c in S:
            if consecvente is not None and c not in consecvente:
                continue
            A = S - {c}
            conf = s / sup[A]
            lift = conf / sup[frozenset([c])]
            if conf >= min_conf and lift > min_lift:
                rez.append({"antecedent": tuple(sorted(A)), "consecvent": c, "support": s, "confidence": conf, "lift": lift})
    return pd.DataFrame(rez, columns=["antecedent", "consecvent", "support", "confidence", "lift"])


def suporturi(freq):
    """DataFrame mlxtend -> dicționar {frozenset(items): suport}."""
    return {frozenset(i): s for i, s in zip(freq["itemsets"], freq["support"])}


sens = []
for s in GRILA_SUPORT:
    fr = fpgrowth(X_tr, min_support=s, use_colnames=True, max_len=MAX_LEN)
    n_reg = len(genereaza_reguli(suporturi(fr), MIN_CONF))
    sens.append({"suport minim": s, "seturi frecvente": len(fr), "din care ≥ 2 produse": int((fr["itemsets"].apply(len) >= 2).sum()),
                 "reguli (conf ≥ 0,5, lift > 1)": n_reg})
sens = pd.DataFrame(sens).set_index("suport minim")
sens.to_csv(ARTEFACTE / "lab5_sensibilitate_suport.csv")
display(sens)
ok = sens[sens["reguli (conf ≥ 0,5, lift > 1)"] <= MAX_REGULI]
MIN_SUPPORT = float(ok.index[0]) if len(ok) else float(sens.index[-1])
print(f"Suport minim ales: {MIN_SUPPORT} ({MIN_SUPPORT * len(X_tr):.0f} coșuri din {len(X_tr):,})")

## 4. Modelare (Modeling)
### 4.1 Seturi frecvente: Apriori, FP-Growth, ECLAT
- **Apriori:** pornește de la itemii frecvenți și construiește seturi tot mai mari, folosind *anti-monotonia suportului* (dacă un set e rar, toate supersețurile lui sunt rare, deci nu le mai generăm).
- **FP-Growth**: comprimă coșurile într-un arbore (FP-Tree) și extrage seturile frecvente din el, fără să genereze candidați.
- **ECLAT**: reprezentare **verticală** (pentru fiecare produs, lista facturilor în care apare); suportul unui set = intersecția listelor. Îl implementăm direct, cu liste de facturi codate ca numere întregi pe biți (intersecția = `&`).
Cele trei trebuie să dea **exact aceleași seturi**; diferă timpul.

In [ ]:
def eclat(X, min_support, max_len):
    """ECLAT: {frozenset(produse): suport}. Fiecare produs = număr întreg cu câte un bit pe factură; intersecția seturilor = `&`."""
    n = len(X)
    min_count = math.ceil(min_support * n - 1e-9)
    biti = {c: int.from_bytes(np.packbits(X[c].to_numpy()).tobytes(), "big") for c in X.columns}
    de_pornire = [(c, b) for c, b in biti.items() if b.bit_count() >= min_count]
    rezultat = {}

    def extinde(prefix, candidati):
        for i, (item, b) in enumerate(candidati):
            nou = prefix + (item,)
            rezultat[frozenset(nou)] = b.bit_count() / n
            if len(nou) < max_len:
                urmatori = [(it2, b & b2) for it2, b2 in candidati[i + 1:] if (b & b2).bit_count() >= min_count]
                if urmatori:
                    extinde(nou, urmatori)

    extinde((), de_pornire)
    return rezultat


timpi, seturi = {}, {}
t0 = time.perf_counter(); seturi["Apriori"] = suporturi(apriori(X_tr, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN)); timpi["Apriori"] = time.perf_counter() - t0
t0 = time.perf_counter(); fr_fp = fpgrowth(X_tr, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN); seturi["FP-Growth"] = suporturi(fr_fp); timpi["FP-Growth"] = time.perf_counter() - t0
t0 = time.perf_counter(); seturi["ECLAT"] = eclat(X_tr, MIN_SUPPORT, MAX_LEN); timpi["ECLAT"] = time.perf_counter() - t0

referinta = set(seturi["Apriori"])
comp_alg = pd.DataFrame({
    "seturi frecvente": {a: len(s) for a, s in seturi.items()},
    "timp (s)": timpi,
    "aceleași seturi ca Apriori": {a: set(s) == referinta for a, s in seturi.items()},
    "diferență maximă de suport": {a: max(abs(s[k] - seturi["Apriori"][k]) for k in referinta & set(s)) for a, s in seturi.items()},
})
comp_alg.to_csv(ARTEFACTE / "lab5_algoritmi.csv")
display(comp_alg)
assert comp_alg["aceleași seturi ca Apriori"].all(), "Algoritmii nu au dat aceleași seturi frecvente"

In [ ]:
sup = seturi["FP-Growth"]
nume = lambda it: " + ".join(f"{descriere.get(c, c)}" for c in sorted(it))
frecvente = pd.DataFrame({"set": [nume(i) for i in sup], "produse": [len(i) for i in sup], "suport": list(sup.values()),
                          "coduri": ["|".join(sorted(i)) for i in sup]})
frecvente = frecvente[frecvente["produse"] >= 2].sort_values("suport", ascending=False)
frecvente.to_csv(ARTEFACTE / "lab5_seturi_frecvente.csv", index=False)
print("Cele mai frecvente seturi cu cel puțin 2 produse:")
display(frecvente.head(10)[["set", "produse", "suport"]])

### 4.2 Reguli de asociere
Din fiecare set frecvent generăm reguli `A ⇒ c` cu un singur produs în consecvent (forma folosită la recomandări) și calculăm support, confidence și lift cu formulele standard. Păstrăm `confidence ≥ 0,5` și `lift > 1` (asociere pozitivă). Restul regulilor (lift ≈ 1 sau < 1) nu spun nimic util: produsele ar apărea împreună la fel de des și întâmplător.

In [ ]:
reguli = genereaza_reguli(sup, MIN_CONF).sort_values("lift", ascending=False).reset_index(drop=True)
reguli["antecedent_text"] = reguli["antecedent"].apply(nume)
reguli["consecvent_text"] = reguli["consecvent"].apply(lambda c: descriere.get(c, c))
print(f"{len(reguli)} reguli | lift: mediană {reguli['lift'].median():.2f}, maxim {reguli['lift'].max():.2f} | confidence: mediană {reguli['confidence'].median():.2f}")
display(reguli.head(10)[["antecedent_text", "consecvent_text", "support", "confidence", "lift"]])

### 4.3 Reguli de asociere cantitative
Regulile cantitative lucrează cu atribute **numerice**, care se **discretizează** în intervale și apoi se rulează același algoritm. Aici fiecare client e o „tranzacție” cu itemii: nivelul de recență, frecvență, valoare medie pe factură, număr de produse, dacă a făcut retururi, țara (UK/altă) și **rezultatul din viitor** (a cumpărat / nu a cumpărat în următoarele 90 de zile). Căutăm reguli care au ca rezultat comportamentul viitor.
**Cum împărțim în intervale?** Există două variante: *binning uniform* sau *pe cuantile*. Comparăm pe `frecventa`: intervale de lățime egală vs. cuantile.

In [ ]:
tabel_binning = pd.DataFrame({
    "lățime egală (3 intervale)": pd.cut(clienti["frecventa"], 3, labels=["mică", "medie", "mare"]).value_counts().sort_index(),
    "cuantile (3 intervale)": pd.qcut(clienti["frecventa"], 3, labels=["mică", "medie", "mare"], duplicates="drop").value_counts().sort_index(),
})
tabel_binning.index.name = "interval"
tabel_binning.to_csv(ARTEFACTE / "lab5_binning.csv")
display(tabel_binning)

In [ ]:
def cuantile(serie, nume_atr):
    return pd.qcut(serie.rank(method="first"), 3, labels=[f"{nume_atr}=mică", f"{nume_atr}=medie", f"{nume_atr}=mare"]).astype(str)


atribute = pd.DataFrame({
    "recență": cuantile(clienti["recenta_zile"], "recență"),
    "frecvență": cuantile(clienti["frecventa"], "frecvență"),
    "valoare/factură": cuantile(clienti["valoare_medie_factura"], "valoare/factură"),
    "produse distincte": cuantile(clienti["produse_distincte"], "produse"),
    "retururi": np.where(clienti["retururi_val"] < 0, "retururi=da", "retururi=nu"),
    "țară": np.where(clienti["uk"] == 1, "țară=UK", "țară=alta"),
    "viitor": np.where(clienti["a_cumparat"], "viitor=cumpără", "viitor=nu cumpără"),
}, index=clienti.index)
tranz_cl = pd.get_dummies(atribute.astype(str)).astype(bool)
tranz_cl.columns = [c.split("_", 1)[1] for c in tranz_cl.columns]
sup_q = suporturi(fpgrowth(tranz_cl, min_support=0.05, use_colnames=True, max_len=4))
reguli_q = genereaza_reguli(sup_q, 0.6, consecvente={"viitor=cumpără", "viitor=nu cumpără"}, min_lift=1.2)
reguli_q["bază"] = reguli_q["consecvent"].map(lambda c: sup_q[frozenset([c])])
reguli_q["antecedent_text"] = reguli_q["antecedent"].apply(lambda a: " ∧ ".join(a))
reguli_q = reguli_q.sort_values("lift", ascending=False).reset_index(drop=True)
reguli_q.drop(columns="antecedent").to_csv(ARTEFACTE / "lab5_reguli_cantitative.csv", index=False)
print(f"{len(reguli_q)} reguli cantitative (confidence ≥ 0,6, lift ≥ 1,2); `bază` = ponderea rezultatului în toți clienții")
display(reguli_q.head(12)[["antecedent_text", "consecvent", "support", "confidence", "lift", "bază"]])

### 4.4 Tipare secvențiale
Secvența unui client = facturile lui în ordine cronologică. Un tipar `{A} → {B}` înseamnă că *A apare într-o factură mai veche decât una în care apare B* (de exemplu `{Laptop} → {Mouse}`). Algoritmi precum AprioriAll, PrefixSpan, SPADE sau GSP extind ideea la secvențe lungi; noi ne oprim la **lungimea 2**, ușor de interpretat.
- Pentru fiecare client și produs reținem ordinea primei și ultimei facturi care îl conține: `A → B` e prezent dacă `prima(A) < ultima(B)`.
- **support** = % din clienții (cu istoric) care au tiparul; **confidence** = `P(B mai târziu | A)`.
- **Contează ordinea?** `confidence` depinde și de cât de populare sunt produsele, deci nu o comparăm între direcții. Folosim un test direct: dintre clienții care au cumpărat ambele produse **în facturi diferite**, ce procent au cumpărat **A primul**? 50% = fără ordine (doar co-apariție); mult peste 50% = A tinde să fie cumpărat înaintea lui B.
- Produse: cele cumpărate de ≥ 5% dintre clienți. Verificăm rezultatul vectorizat cu o numărare directă pe câteva perechi.

In [ ]:
inv = istoric_v[["Customer ID", "Invoice", "InvoiceDate", "StockCode"]].drop_duplicates(["Invoice", "StockCode"]).copy()
inv["ord"] = inv.groupby("Customer ID")["InvoiceDate"].rank(method="dense")
cl_per_produs = inv.groupby("StockCode")["Customer ID"].nunique()
n_cl = inv["Customer ID"].nunique()
prod_seq = cl_per_produs[cl_per_produs >= 0.05 * n_cl].index
inv = inv[inv["StockCode"].isin(prod_seq)]
prima = inv.groupby(["Customer ID", "StockCode"])["ord"].min().unstack().reindex(columns=prod_seq)
ultima = inv.groupby(["Customer ID", "StockCode"])["ord"].max().unstack().reindex(columns=prod_seq)
Fp, Lp = prima.to_numpy(float), ultima.to_numpy(float)                 # NaN = clientul nu a cumpărat produsul
n_a = (~np.isnan(Fp)).sum(axis=0)

cnt = np.zeros((len(prod_seq), len(prod_seq)))                          # clienți cu prima(A) < ultima(B)
primul = np.zeros_like(cnt)                                              # clienți cu prima(A) < prima(B)
for a in range(len(prod_seq)):
    cnt[a] = (Fp[:, [a]] < Lp).sum(axis=0)                             # NaN < x este False
    primul[a] = (Fp[:, [a]] < Fp).sum(axis=0)
np.fill_diagonal(cnt, 0); np.fill_diagonal(primul, 0)

# verificare cu numărare directă pe 5 perechi alese la întâmplare
rng = np.random.default_rng(SEED)
for a, b in rng.choice(len(prod_seq), (5, 2)):
    if a == b:
        continue
    direct, direct_primul = 0, 0
    for _, g in inv[inv["StockCode"].isin([prod_seq[a], prod_seq[b]])].groupby("Customer ID"):
        oa, ob = g.loc[g["StockCode"] == prod_seq[a], "ord"], g.loc[g["StockCode"] == prod_seq[b], "ord"]
        if len(oa) > 0 and len(ob) > 0:
            direct += int(oa.min() < ob.max())
            direct_primul += int(oa.min() < ob.min())
    assert (direct, direct_primul) == (cnt[a, b], primul[a, b]), (prod_seq[a], prod_seq[b], direct, cnt[a, b], direct_primul, primul[a, b])
print("Verificare numărare directă: OK")

ambele = primul + primul.T                                               # clienți cu ambele produse, în facturi diferite (prima(A) ≠ prima(B))
ia, ib = np.where((primul >= primul.T) & (ambele >= 30) & (primul > primul.T))   # fiecare pereche apare o singură dată, în direcția dominantă
secv = pd.DataFrame({"A": prod_seq[ia], "B": prod_seq[ib], "clienți cu ambele": ambele[ia, ib], "A primul %": primul[ia, ib] / ambele[ia, ib] * 100,
                     "clienți A→B": cnt[ia, ib], "clienți cu A": n_a[ia]})
secv["support"] = secv["clienți A→B"] / n_cl
secv["confidence"] = secv["clienți A→B"] / secv["clienți cu A"]
secv["produs A"], secv["produs B"] = descriere.reindex(secv["A"]).values, descriere.reindex(secv["B"]).values
secv = secv.sort_values("A primul %", ascending=False)
secv.to_csv(ARTEFACTE / "lab5_secventiale.csv", index=False)
print(f"Produse analizate (cumpărate de ≥ 5% dintre clienți): {len(prod_seq)} | clienți: {n_cl:,} | perechi cu ≥ 30 de clienți și o ordine dominantă: {len(secv):,}")
print("Distribuția lui „A primul %” (50% = fără ordine):")
display(secv["A primul %"].describe().to_frame().T)
print("Perechile cu cea mai clară ordine:")
display(secv.head(10)[["produs A", "produs B", "clienți cu ambele", "A primul %", "support", "confidence"]])

## 5. Evaluare (Evaluation)
### 5.1 Rezistă regulile în timp?
Pentru fiecare regulă găsită pe **istoric**, recalculăm support, confidence și lift pe coșurile **de după data de tăiere**. Ne uităm la: câte reguli păstrează lift > 1 și confidence ≥ 0,5 și cât de bine se corelează lift-ul din istoric cu cel din test (corelația Spearman).

In [ ]:
def metrici_pe(X, reguli_df):
    """Support/confidence/lift ale regulilor pe un alt set de coșuri."""
    v = X.to_numpy()
    col = {c: i for i, c in enumerate(X.columns)}
    out = []
    for ant, cons in zip(reguli_df["antecedent"], reguli_df["consecvent"]):
        ma = v[:, [col[c] for c in ant]].all(axis=1)
        mc = v[:, col[cons]]
        sa, sc, sac = ma.mean(), mc.mean(), (ma & mc).mean()
        conf = sac / sa if sa > 0 else np.nan
        out.append((sac, conf, conf / sc if sc > 0 and sa > 0 else np.nan))
    return pd.DataFrame(out, columns=["support_test", "confidence_test", "lift_test"], index=reguli_df.index)


reguli = reguli.join(metrici_pe(X_te, reguli))
stabil = pd.DataFrame({"valoare": [
    len(reguli), f"{(reguli['lift_test'] > 1).mean():.1%}", f"{((reguli['lift_test'] > 1) & (reguli['confidence_test'] >= MIN_CONF)).mean():.1%}",
    f"{reguli[['lift', 'lift_test']].corr(method='spearman').iloc[0, 1]:.3f}", f"{(reguli['confidence_test'] - reguli['confidence']).median():+.3f}",
    f"{(reguli['confidence_test'] - reguli['confidence']).abs().median():.3f}"]},
    index=["reguli evaluate", "păstrează lift > 1 în test", "păstrează lift > 1 și confidence ≥ 0,5", "corelația Spearman lift istoric vs. test",
           "schimbarea mediană a confidence (test − istoric)", "schimbarea absolută mediană a confidence"])
stabil.to_csv(ARTEFACTE / "lab5_stabilitate.csv")
display(stabil)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sc = ax[0].scatter(reguli["support"], reguli["confidence"], c=reguli["lift"], s=10, cmap="viridis")
plt.colorbar(sc, ax=ax[0], label="lift"); ax[0].set_xlabel("support"); ax[0].set_ylabel("confidence"); ax[0].set_title("Regulile găsite (istoric)")
ax[1].scatter(reguli["lift"], reguli["lift_test"], s=10, alpha=0.6)
lim = [0, max(reguli["lift"].max(), reguli["lift_test"].max(skipna=True)) * 1.05]
ax[1].plot(lim, lim, "k--"); ax[1].axhline(1, color="gray", lw=0.7)
ax[1].set_xlabel("lift în istoric"); ax[1].set_ylabel("lift în test"); ax[1].set_title("Stabilitate în timp")
plt.tight_layout()
salveaza_fig("lab5_reguli")

### 5.2 Sunt utile ca recomandări?
Test „ascunde și ghicește” pe coșurile din perioada de test: din fiecare coș cu cel puțin 2 produse (din cele păstrate) ascundem un produs ales la întâmplare și recomandăm 5 produse pe baza celorlalte. **Hit rate@5** = % din coșuri în care produsul ascuns e printre cele 5 recomandate. **Acoperire** = % din coșuri în care regulile pot recomanda ceva.
Comparăm trei strategii: reguli ordonate după **lift**, reguli ordonate după **confidence** și **baza** (cele 5 produse cele mai populare din istoric, care nu sunt deja în coș).

In [ ]:
def recomanda(cos, lista_reguli, n=5):
    rec = []
    for ant, cons in lista_reguli:
        if cons not in cos and cons not in rec and ant <= cos:
            rec.append(cons)
            if len(rec) == n:
                break
    return rec


lista_lift = [(frozenset(a), c) for a, c in zip(reguli["antecedent"], reguli["consecvent"])]
ordine_conf = reguli.sort_values(["confidence", "lift"], ascending=False)
lista_conf = [(frozenset(a), c) for a, c in zip(ordine_conf["antecedent"], ordine_conf["consecvent"])]
populare = list(X_tr.sum().sort_values(ascending=False).index)

coloane = np.array(X_te.columns)
valori = X_te.to_numpy()
candidati = np.flatnonzero(valori.sum(axis=1) >= 2)
rng = np.random.default_rng(SEED)
esantion = rng.choice(candidati, size=min(5000, len(candidati)), replace=False)
rezultate = {"reguli (după lift)": [0, 0], "reguli (după confidence)": [0, 0], "baza: populare": [0, 0]}
for i in esantion:
    cos = set(coloane[valori[i]])
    tinta = rng.choice(sorted(cos))
    ramas = frozenset(cos - {tinta})
    for nume_s, rec in [("reguli (după lift)", recomanda(ramas, lista_lift)), ("reguli (după confidence)", recomanda(ramas, lista_conf)),
                        ("baza: populare", [p for p in populare if p not in ramas][:5])]:
        rezultate[nume_s][0] += int(tinta in rec)
        rezultate[nume_s][1] += int(len(rec) > 0)
recomandari = pd.DataFrame({"hit rate@5 %": {k: v[0] / len(esantion) * 100 for k, v in rezultate.items()},
                            "acoperire %": {k: v[1] / len(esantion) * 100 for k, v in rezultate.items()}})
recomandari.to_csv(ARTEFACTE / "lab5_recomandari.csv")
print(f"Coșuri evaluate: {len(esantion):,} (din {len(candidati):,} coșuri de test cu ≥ 2 produse păstrate)")
display(recomandari)

### 5.3 Tabelul regulilor, cu și fără validare în timp
Primele reguli după lift, cu valorile din istoric și din test lângă ele. **Cum citim:** lift mare în istoric dar apropiat de 1 în test = regulă care nu s-a păstrat; suportul mic face valorile instabile.
**Atenție la utilitatea practică:** reguli între variante ale aceluiași produs (culori/modele dintr-o serie) au lift mare, dar sunt evidente; ele nu aduc informație nouă pentru recomandări.

In [ ]:
afis = reguli.head(15)[["antecedent_text", "consecvent_text", "support", "confidence", "lift", "support_test", "confidence_test", "lift_test"]]
display(afis)

## 6. Implementare (Deployment)
Salvăm regulile (cu codurile produselor, pentru a le potrivi pe un coș), seturile frecvente și tiparele secvențiale. Aplicația Streamlit recomandă produse pentru un coș ales de utilizator, cu aceeași logică de la 5.2.

In [ ]:
salv = reguli.copy()
salv["antecedent_cod"] = salv["antecedent"].apply(lambda a: "|".join(a))
salv.drop(columns="antecedent").to_csv(ARTEFACTE / "lab5_reguli.csv", index=False)
pd.DataFrame({"StockCode": X_tr.columns, "Description": descriere.reindex(X_tr.columns).values, "suport": X_tr.mean().values}).to_csv(ARTEFACTE / "lab5_produse.csv", index=False)

info5 = {
    "coșuri_train": int(len(X_tr)), "coșuri_test": int(len(X_te)), "produse": int(onehot.shape[1]), "suport_minim": MIN_SUPPORT,
    "confidence_minim": MIN_CONF, "max_len": MAX_LEN, "reguli": int(len(reguli)), "seturi_frecvente": int(len(sup)),
    "reguli_cantitative": int(len(reguli_q)), "perechi_secventiale": int(len(secv)), "data_taiere": str(CUTOFF.date()),
    "hit_rate_lift": float(recomandari.loc["reguli (după lift)", "hit rate@5 %"]), "hit_rate_baza": float(recomandari.loc["baza: populare", "hit rate@5 %"]),
    "lift_test_pastrat": float((reguli["lift_test"] > 1).mean()),
}
(ARTEFACTE / "info_lab5.json").write_text(json.dumps(info5, ensure_ascii=False, indent=2), encoding="utf-8")

# test: recomandări pentru un coș-exemplu (primul antecedent al primei reguli)
cos_ex = frozenset(reguli.loc[0, "antecedent"])
print("Coș-exemplu:", nume(cos_ex))
print("Recomandare:", [descriere.get(c, c) for c in recomanda(cos_ex, lista_lift, 3)])

## 7. Concluzii și limite
- Pragurile (suport minim, confidence ≥ 0,5, lift > 1) sunt alese cu cifre (3, tabelul de sensibilitate); algoritmii dau aceleași seturi (4.1).
- Regulile se judecă pe date noi (5.1) și ca recomandări față de o bază (5.2), nu doar după lift în istoric.
- Regulile cantitative descriu clienți (4.3), dar nu sunt un clasificator: pentru predicție vezi Laboratorul 2.
- Tiparele secvențiale (4.4) au lungime 2; „A primul %” arată dacă ordinea contează (mult peste 50%) sau e doar co-apariție (≈ 50%).
- **Limite:** produse cu suport sub pragul ales nu apar în reguli (coada lungă); regulile între variante ale aceleiași serii sunt evidente; coșurile dintr-o singură factură nu spun nimic despre intenția clientului; fereastra de test include sezonul sărbătorilor.